# Data preparation and EDA

Run to inventory source files, create the stratified analysis sample, and save basic data-quality summaries and plots.

In [ ]:
from collections import Counter
from pathlib import Path
import csv
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "raw_data").is_dir() and (p / "README.md").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root containing raw_data/ and README.md.")

RAW_DIR = PROJECT_ROOT / "raw_data"
OUTPUT_DIR = PROJECT_ROOT / "processed_data"
FIGURE_DIR = OUTPUT_DIR / "figures"
TARGET_SAMPLE_SIZE = 50_000
CHUNK_SIZE = 100_000
RANDOM_STATE = 42
FIELDS = [
    "Program", "LoanStatus", "BusinessAge", "TermInMonths", "GrossApproval",
    "JobsSupported", "CollateralInd", "NaicsCode", "FranchiseCode", "RevolverStatus",
]
CSV_FILES = sorted(RAW_DIR.glob("FOIA_*.csv"))
if not CSV_FILES:
    raise FileNotFoundError(f"No SBA FOIA CSV files found in {RAW_DIR}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def normalize_status(value):
    if pd.isna(value):
        return None
    status = re.sub(r"\s+", "", str(value).strip().upper())
    return {"CHGOFF": 1, "PIF": 0}.get(status)

def normalize_program(value):
    if pd.isna(value):
        return ""
    program = str(value).strip().upper()
    return "7(a)" if program in {"7A", "7(A)"} else program

def read_chunks(path):
    with path.open("r", encoding="utf-8-sig", newline="") as stream:
        columns = next(csv.reader(stream))
    missing = {"Program", "LoanStatus"}.difference(columns)
    if missing:
        raise ValueError(f"{path.name} is missing required fields: {sorted(missing)}")
    use_columns = [column for column in FIELDS if column in columns]
    return pd.read_csv(
        path, usecols=use_columns, dtype="string", encoding="utf-8-sig",
        chunksize=CHUNK_SIZE, low_memory=False,
    )

def prepare_chunk(chunk):
    chunk = chunk.copy()
    chunk["Program"] = chunk["Program"].map(normalize_program)
    chunk["Default"] = chunk["LoanStatus"].map(normalize_status)
    chunk = chunk.loc[chunk["Default"].notna() & chunk["Program"].ne("")].copy()
    chunk["Default"] = chunk["Default"].astype("int8")
    chunk["LoanStatus"] = chunk["Default"].map({1: "CHGOFF", 0: "PIF"})
    for column in ("TermInMonths", "GrossApproval", "JobsSupported"):
        if column in chunk:
            chunk[column] = pd.to_numeric(chunk[column], errors="coerce")
    for column in ("BusinessAge", "CollateralInd", "NaicsCode", "FranchiseCode", "RevolverStatus"):
        if column in chunk:
            chunk[column] = chunk[column].str.strip().replace("", pd.NA)
    if "CollateralInd" in chunk:
        chunk["CollateralInd"] = chunk["CollateralInd"].str.upper()
    if "RevolverStatus" in chunk:
        chunk["RevolverStatus"] = chunk["RevolverStatus"].replace({"0": "Term Loan", "1": "Revolver"})
    if "NaicsCode" in chunk:
        chunk["NaicsSector"] = chunk["NaicsCode"].str.extract(r"^(\d{2})", expand=False)
        chunk = chunk.drop(columns="NaicsCode")
    if "FranchiseCode" in chunk:
        franchise = chunk["FranchiseCode"].str.upper()
        chunk["FranchiseInd"] = franchise.notna() & ~franchise.isin({"0", "0.0", "NONE", "N/A"})
        chunk["FranchiseInd"] = chunk["FranchiseInd"].astype("int8")
        chunk = chunk.drop(columns="FranchiseCode")
    return chunk

def proportional_quotas(counts, sample_size):
    total = sum(counts.values())
    if total < sample_size:
        raise ValueError(f"Only {total:,} eligible records; cannot draw {sample_size:,} without replacement.")
    exact = {key: sample_size * count / total for key, count in counts.items()}
    quotas = {key: int(value) for key, value in exact.items()}
    remaining = sample_size - sum(quotas.values())
    for key in sorted(exact, key=lambda item: exact[item] - quotas[item], reverse=True)[:remaining]:
        quotas[key] += 1
    return quotas

print(f"Found {len(CSV_FILES)} input CSVs; processing in chunks of {CHUNK_SIZE:,}.")

In [ ]:
stratum_counts = Counter()
source_inventory = []
for path in CSV_FILES:
    with path.open("r", encoding="utf-8-sig", newline="") as stream:
        headers = next(csv.reader(stream))
    source_inventory.append({"file": path.name, "size_mb": round(path.stat().st_size / 1_000_000, 2),
                             "column_count": len(headers)})
    for chunk in read_chunks(path):
        prepared = prepare_chunk(chunk)
        stratum_counts.update(prepared.groupby(["Program", "LoanStatus"], observed=True).size().to_dict())

inventory = pd.DataFrame(source_inventory)
counts = pd.DataFrame([{"Program": key[0], "LoanStatus": key[1], "eligible_count": value}
                       for key, value in stratum_counts.items()])
if not stratum_counts:
    raise ValueError("No paid-in-full or charged-off records were found after status normalization.")
sample_size = min(TARGET_SAMPLE_SIZE, sum(stratum_counts.values()))
quotas = proportional_quotas(stratum_counts, sample_size)
print(f"Eligible records: {sum(stratum_counts.values()):,}; planned sample drawn: {sample_size:,}")
display(inventory, counts)

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
reservoir = {}
for path in CSV_FILES:
    for chunk in read_chunks(path):
        prepared = prepare_chunk(chunk)
        prepared["_sample_key"] = rng.random(len(prepared))
        for key, group in prepared.groupby(["Program", "LoanStatus"], observed=True, sort=False):
            quota = quotas.get(key, 0)
            if quota == 0:
                continue
            candidates = pd.concat([reservoir.get(key, group.iloc[0:0]), group], ignore_index=True)
            reservoir[key] = candidates.nsmallest(quota, "_sample_key")

sample = pd.concat(reservoir.values(), ignore_index=True).drop(columns="_sample_key")
if len(sample) != sample_size:
    raise RuntimeError(f"Expected {sample_size:,} sampled rows; got {len(sample):,}.")
sample_path = OUTPUT_DIR / "analysis_sample.csv"
sample.to_csv(sample_path, index=False)

quality = pd.DataFrame({
    "missing_count": sample.isna().sum(),
    "missing_percent": (sample.isna().mean() * 100).round(2),
    "dtype": sample.dtypes.astype(str),
})
status_counts = sample.groupby(["Program", "LoanStatus"], observed=True).size().rename("count").reset_index()
age_outcome = pd.crosstab(sample["BusinessAge"], sample["LoanStatus"], dropna=False)
quality.to_csv(OUTPUT_DIR / "data_quality_summary.csv")
status_counts.to_csv(OUTPUT_DIR / "sample_outcome_counts.csv", index=False)
age_outcome.to_csv(OUTPUT_DIR / "business_age_by_outcome.csv")
print(f"Saved {len(sample):,} rows to {sample_path}")
display(quality, status_counts, age_outcome)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.countplot(data=sample, x="LoanStatus", hue="Program", ax=axes[0])
axes[0].set(title="Sample counts by loan outcome and program", xlabel="Loan status", ylabel="Loans")
sns.countplot(data=sample, y="BusinessAge", hue="LoanStatus", ax=axes[1])
axes[1].set(title="Business age by loan outcome", xlabel="Loans", ylabel="Business age")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "interim_eda_overview.png", dpi=160, bbox_inches="tight")
plt.show()

if "TermInMonths" in sample:
    term_summary = sample.groupby("LoanStatus", observed=True)["TermInMonths"].describe()
    term_summary.to_csv(OUTPUT_DIR / "term_by_outcome_summary.csv")
    display(term_summary)